# A pressure mat that answers questions in English

A bed pressure mat streams a grid of numbers. You cannot read it; the model can.

This notebook loads [Tactus Mat](https://huggingface.co/EximiusLabs/fusion-embedding-2-tactus-mat)
and real held-out frames from the [PhysioNet pressure map dataset](https://physionet.org/content/pmd/1.0.0/),
then lets you **type any sentence** and find the matching moment. There is no posture
classifier anywhere in here: recognition is a cosine ranking in a shared text embedding space,
so a phrase nobody trained on works exactly as well as one that was.

Runtime: pick a GPU (Runtime -> Change runtime type -> T4). CPU works but the text side is slow.

In [ ]:
%pip install -q "fusion-embedding[hf]" huggingface_hub safetensors matplotlib

In [ ]:
import importlib.util, sys
import numpy as np, torch
from huggingface_hub import hf_hub_download

REPO, REV = "EximiusLabs/fusion-embedding-2-tactus-mat", "v0.1-preview"

# the model's own inference code, straight from the repo
for f in ("inference.py", "tactile.py"):
    path = hf_hub_download(REPO, f, revision=REV)
sys.path.insert(0, __import__("os").path.dirname(path))
spec = importlib.util.spec_from_file_location("tm_inf", hf_hub_download(REPO, "inference.py", revision=REV))
tm_inf = importlib.util.module_from_spec(spec); spec.loader.exec_module(tm_inf)

device = "cuda" if torch.cuda.is_available() else "cpu"
tm = tm_inf.TactusMatEmbedder.from_pretrained(REPO, revision=REV, device=device,
                                              dtype=torch.float16 if device == "cuda" else torch.float32)

d = np.load(hf_hub_download(REPO, "demo_frames.npz", revision=REV))
names = [str(n) for n in d["position_names"]]
print(f"model on {device}; {len(d['gallery'])} gallery windows, {len(d['night'])} night frames")

## 1. What the mat sees, and what the model says

Eight real windows from a subject the model never trained on. Left: the raw pressure map.
Right: the model's top-3 out of the 17 posture phrases.

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F

phrases = [f"a person {n}, recorded by a bed pressure mat" for n in names]
T = F.normalize(tm.embed_text(phrases), dim=-1)          # [17, 2048], computed once
G = torch.stack([tm.embed_pressure(w.astype(np.float32)) for w in d["gallery"]])

fig, axes = plt.subplots(2, 8, figsize=(15, 5), gridspec_kw={"height_ratios": [3, 1]})
for i, (win, p) in enumerate(zip(d["gallery"], d["gallery_pos"])):
    sims = (T @ G[i]).numpy()
    axes[0, i].imshow(win.astype(np.float32).max(0), cmap="magma", vmin=0, vmax=0.6, aspect="auto")
    axes[0, i].set_xticks([]); axes[0, i].set_yticks([])
    axes[0, i].set_title(f"truth: {names[p-1][:22]}", fontsize=6.5)
    top = np.argsort(-sims)[:3]
    axes[1, i].barh(range(3)[::-1], sims[top] + 0.2,
                    color=["#C96F4A" if t == p-1 else "#D9CDBA" for t in top])
    for j, t in enumerate(top):
        axes[1, i].text(0.01, 2-j, names[t][:24], fontsize=5, va="center", color="#1F3A5F")
    axes[1, i].set_xticks([]); axes[1, i].set_yticks([])
    [s.set_visible(False) for s in axes[1, i].spines.values()]
plt.suptitle("orange = the recorded posture", fontsize=9, color="#1F3A5F"); plt.tight_layout(); plt.show()

## 2. Type your own sentence

This is the part a classifier cannot do. Write anything, including words that appear in no
label and no training phrase, and the model ranks the real mat windows against it.

Verified to work: `a person starfished across the bed` (nothing in training says "starfished"),
`both knees raised`, `knees pulled up toward the chest`, `arms and legs spread wide`,
`lying on the right`, `a person on their left side`.

It is not magic, and the misses are worth seeing too. `the fetal position` works, but
`someone curled up` lands on a supine window and `someone sleeping on the left` flips to the
right side. The model matches how a body presses into a mat, so a phrasing that leans on
connotation rather than posture can miss.

In [ ]:
QUERY = "a person starfished across the bed"      #  <-- edit me and re-run

q = F.normalize(tm.embed_text([QUERY]), dim=-1)[0]
scores = (G @ q).numpy()
order = np.argsort(-scores)

fig, axes = plt.subplots(1, 4, figsize=(8, 3.2))
for ax, i in zip(axes, order[:4]):
    ax.imshow(d["gallery"][i].astype(np.float32).max(0), cmap="magma", vmin=0, vmax=0.6, aspect="auto")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{scores[i]:+.3f}\n{names[d['gallery_pos'][i]-1][:26]}", fontsize=7)
plt.suptitle(f'best matches for: "{QUERY}"', fontsize=10, color="#1F3A5F")
plt.tight_layout(); plt.show()

## 3. Ask a question about time

The mat streams at 1 Hz, so posture is a time series and "when did they last move?" is a
query rather than a chart you read by eye. The sequence below is real recording from one
held-out subject, sessions concatenated in a fixed order.

In [ ]:
COARSE = ["a person lying flat on their back",
          "a person lying on their right side",
          "a person lying on their left side"]
C = F.normalize(tm.embed_text(COARSE), dim=-1)

night = d["night"].astype(np.float32)
emb = torch.stack([tm.embed_pressure(night[i:i+1]) for i in range(len(night))])
pred = (emb @ C.T).argmax(1).numpy()

w = 9                                     # majority filter, 9 seconds
sm = np.array([np.bincount(pred[max(0, i-w//2):i+w//2+1], minlength=3).argmax()
               for i in range(len(pred))])
changes = np.where(np.diff(sm) != 0)[0] + 1

plt.figure(figsize=(9, 1.6))
for c, col in enumerate(["#1F3A5F", "#C96F4A", "#7A9E7E"]):
    plt.fill_between(np.arange(len(sm)), 0, 1, where=sm == c, step="mid", color=col)
plt.yticks([]); plt.xlabel("seconds"); plt.title("posture over the session", fontsize=9)
plt.show()

print(f"repositioning events detected: {len(changes)} at t = {changes.tolist()} s")
print(f"last movement: t = {int(changes[-1])} s, {len(sm)-int(changes[-1])} s ago")
print(f"currently: {COARSE[sm[-1]]}")

## Your own mat

The model takes `[F, 64, 32]` frames scaled to `[0,1]` (raw counts go in with `raw="fsa"`).
A different mat geometry needs a short fine-tune, not a new foundation model: the language
side stays frozen and only a ~16M-parameter head is trained, which is about half an hour on
one GPU. Recipe and training code: [github.com/Eximius-Labs/fusion-embedding](https://github.com/Eximius-Labs/fusion-embedding).

Numbers, licensing (ODC-By, attribution to Pouyan et al. 2017 and PhysioNet), and limits are
on the [model card](https://huggingface.co/EximiusLabs/fusion-embedding-2-tactus-mat).
**Not a medical device.**